# Demo 10-12 min - Storytelling tecnico
## Deteccion de outliers en valor unitario por subpartida

**Tesis:** Deteccion de variaciones en el valor unitario mediante algoritmos de aprendizaje no supervisado en las exportaciones peruanas.

**Idea central de la demo:**  
La unidad de analisis no es toda la base mezclada. Primero se divide el universo por `NUM_PARTNANDI` y, dentro de cada subpartida, se aplican folds, train/test, deteccion de outliers y comentarios tecnicos.

**Estructura de exposicion:**

1. Problema y metrica.
2. Protocolo: subpartida, folds, seeds y anti-leakage.
3. Resultados: tabla principal y una figura clave.
4. Ablaciones: que cambio movio la aguja.
5. Riesgos y plan.
6. Reproducibilidad: comando, artefactos, MLflow opcional y repo.

## 0. Antes del codigo: decision metodologica

Este notebook trabaja con dos escenarios:

**Escenario A - Datos reales sin etiqueta:**  
Sirve para detectar registros atipicos y generar una tabla de revision. Como no existe etiqueta real confiable, no se reporta accuracy como metrica principal.

**Escenario B - Datos reales + outliers sinteticos:**  
Sirve para validar si el detector recupera outliers conocidos. La data sintetica no reemplaza la data real; solo permite medir `precision`, `recall` y `f1` sobre anomalias insertadas de forma controlada.

Regla principal:

```text
Base de exportaciones
    -> dividir por NUM_PARTNANDI
        -> dentro de cada subpartida crear folds
            -> train real
            -> test real + outliers sinteticos conocidos
            -> evaluar recuperacion
```

## 1. Problema y metrica

El problema no es afirmar fraude automaticamente. El objetivo es detectar **variaciones atipicas del valor unitario** en exportaciones peruanas para priorizar revision tecnica.

Formula principal:

```text
VALOR_UNITARIO = FOB_DOLAR / PESO_NETO
```

Metricas usadas:

- En datos reales: cantidad de alertas, tasa de alertas, limites aprendidos, ranking por severidad.
- En datos con outliers sinteticos: precision, recall y f1.
- En comparacion por folds: estabilidad promedio y desviacion.

In [1]:
# =============================================================================
# 1. CONFIGURACION GENERAL
# =============================================================================

from __future__ import annotations

import json
import math
import os
import time
import warnings
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Any
from urllib.parse import quote_plus

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import KFold
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# Semilla reproducible
# -----------------------------------------------------------------------------
SEED = 42
RNG = np.random.default_rng(SEED)

# -----------------------------------------------------------------------------
# Fuente unica de datos: SQL Server
# No se carga TXT ni CSV en este notebook.
# -----------------------------------------------------------------------------
FUENTE_DATOS = "SQL_SERVER"

# -----------------------------------------------------------------------------
# Conexion SQL Server
# Ajustar estos valores segun tu entorno Windows.
# -----------------------------------------------------------------------------
SERVIDOR = r"DESKTOP-OGU19A7\SQLEXPRESS,56878"
BASE_DATOS = "DB_GEE_DW_ADUANAS"
ESQUEMA = "SC_ADUANA"
PROCEDIMIENTO = "SP_VALORES_UNITARIOS"
DRIVER = "ODBC Driver 17 for SQL Server"

FEC_INI = "2024-01-01"
FEC_FIN = "2024-12-31"

# -----------------------------------------------------------------------------
# Columnas canonicas del notebook
# -----------------------------------------------------------------------------
COL_SUBPARTIDA = "NUM_PARTNANDI"
COL_FOB = "FOB_DOLAR"
COL_PESO = "PESO_NETO"
COL_CANTIDAD = "CANTIDAD_EXPORTADA"
COL_FECHA = "FECHA_EXPORTACION"
COL_OPERADOR = "RUC_EXPORTADOR"
COL_ADUANA = "ADUANA"
COL_VU = "VALOR_UNITARIO"
COL_LOG_VU = "LOG_VALOR_UNITARIO"

# -----------------------------------------------------------------------------
# Columnas candidatas para adaptar la salida real del procedimiento SQL Server
# -----------------------------------------------------------------------------
MAPEO_CANDIDATAS = {
    COL_SUBPARTIDA: ["NUM_PARTNANDI", "NUM_SPN_R", "SUBPARTIDA", "PART_NANDI", "PARTIDA_NANDINA"],
    COL_FOB: ["FOB_DOLAR", "MTO_FOBDOL", "MTO_FOB_DOLAR", "FOB", "VALOR_FOB"],
    COL_PESO: ["PESO_NETO", "CNT_PESO_NETO", "PESO_NETO_KG", "PESO_KG"],
    COL_CANTIDAD: ["CANTIDAD_EXPORTADA", "CNT_UNIFIS", "CANTIDAD", "UNIDADES"],
    COL_FECHA: ["FECHA_EXPORTACION", "FEC_NUM", "FECHA", "FEC_DECLARACION", "FECHA_DECLARACION"],
    COL_OPERADOR: ["RUC_EXPORTADOR", "NUM_RUC", "RUC", "EXPORTADOR"],
    COL_ADUANA: ["ADUANA", "COD_ADUANA", "DESC_ADUANA"],
}

# -----------------------------------------------------------------------------
# Parametros de demo
# -----------------------------------------------------------------------------
MIN_REGISTROS_SUBPARTIDA = 40
N_SUBPARTIDAS_DEMO = 5
N_FOLDS = 5
TASA_OUTLIERS_SINTETICOS = 0.08
FACTOR_OUTLIER_BAJO = 0.08
FACTOR_OUTLIER_ALTO = 6.00

# Si deseas forzar codigos especificos, coloca aqui una lista.
# Ejemplo: SUBPARTIDAS_DEMO_FORZADAS = ["0806100000", "0804400000"]
SUBPARTIDAS_DEMO_FORZADAS: list[str] = []

# Capitulos agropecuarios referenciales para demo: 06, 07, 08, 09, 10, 12.
CAPITULOS_AGRO_DEMO = ("06", "07", "08", "09", "10", "12")

# -----------------------------------------------------------------------------
# Salidas
# Todos los CSV/TXT se exportan con delimitador pipe.
# -----------------------------------------------------------------------------
ARTIFACTS_DIR = Path("artifacts_demo_storytelling_sql_server")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTADOS_CSV = ARTIFACTS_DIR / "resultados_modelos_por_subpartida.csv"
ABLACIONES_CSV = ARTIFACTS_DIR / "ablaciones_global_vs_subpartida.csv"
ALERTAS_CSV = ARTIFACTS_DIR / "alertas_reales_priorizadas.csv"
README_TXT = ARTIFACTS_DIR / "README_REPRODUCIBILIDAD.txt"
FIGURA_CLAVE = ARTIFACTS_DIR / "figura_clave_outliers_por_subpartida.png"

print("Configuracion cargada.")
print(f"Fuente de datos SQL Server: {FUENTE_DATOS}")
print(f"SQL Server: {SERVIDOR}")
print(f"Base de datos: {BASE_DATOS}")
print(f"Procedimiento: {ESQUEMA}.{PROCEDIMIENTO}")
print(f"Periodo: {FEC_INI} a {FEC_FIN}")
print(f"Carpeta de artefactos: {ARTIFACTS_DIR.resolve()}")

Configuracion cargada.
Fuente de datos SQL Server: SQL_SERVER
SQL Server: DESKTOP-OGU19A7\SQLEXPRESS,56878
Base de datos: DB_GEE_DW_ADUANAS
Procedimiento: SC_ADUANA.SP_VALORES_UNITARIOS
Periodo: 2024-01-01 a 2024-12-31
Carpeta de artefactos: C:\Users\hp\Downloads\ROBOTICA\TESIS_2\notebooks\04_parcial\artifacts_demo_storytelling_sql_server


## 2. Funciones de carga desde SQL Server y estandarizacion

Esta seccion conecta directamente a SQL Server usando el procedimiento configurado.  
No se carga TXT ni CSV.

La columna principal de agrupacion sera siempre `NUM_PARTNANDI`.  
Si el procedimiento devuelve `NUM_SPN_R`, el notebook la renombra automaticamente a `NUM_PARTNANDI`.

In [2]:
# =============================================================================
# 2. FUNCIONES DE CARGA SQL SERVER, ESTANDARIZACION Y VALIDACION
# =============================================================================

def normalizar_nombre_columna(nombre: str) -> str:
    """
    Normaliza un nombre de columna para comparaciones robustas.

    Parameters
    ----------
    nombre : str
        Nombre original de la columna.

    Returns
    -------
    str
        Nombre normalizado en mayusculas, sin espacios laterales.
    """
    return str(nombre).strip().upper()


def buscar_columna(df: pd.DataFrame, candidatas: list[str]) -> str | None:
    """
    Busca la primera columna existente dentro de una lista de candidatas.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset original.
    candidatas : list[str]
        Posibles nombres de la columna.

    Returns
    -------
    str | None
        Nombre real encontrado en el DataFrame. Retorna None si no existe.
    """
    columnas_norm = {normalizar_nombre_columna(col): col for col in df.columns}
    for candidata in candidatas:
        clave = normalizar_nombre_columna(candidata)
        if clave in columnas_norm:
            return columnas_norm[clave]
    return None


def estandarizar_columnas(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Estandariza el dataset hacia columnas canonicas para la demo.

    Parameters
    ----------
    df_raw : pd.DataFrame
        Dataset crudo cargado desde SQL Server.

    Returns
    -------
    pd.DataFrame
        Dataset con columnas canonicas disponibles.

    Raises
    ------
    ValueError
        Si faltan columnas esenciales: subpartida, FOB y peso neto.
    """
    df = df_raw.copy()
    renombres = {}

    for canonica, candidatas in MAPEO_CANDIDATAS.items():
        encontrada = buscar_columna(df, candidatas)
        if encontrada is not None:
            renombres[encontrada] = canonica

    df = df.rename(columns=renombres)

    requeridas = [COL_SUBPARTIDA, COL_FOB, COL_PESO]
    faltantes = [col for col in requeridas if col not in df.columns]
    if faltantes:
        raise ValueError(
            "Faltan columnas esenciales para la demo: "
            f"{faltantes}. Columnas disponibles: {list(df_raw.columns)}"
        )

    return df


def cargar_datos_sql_server() -> pd.DataFrame:
    """
    Carga datos desde SQL Server mediante procedimiento almacenado.

    Parameters
    ----------
    None
        Usa las constantes globales de conexion y periodo.

    Returns
    -------
    pd.DataFrame
        Dataset original extraido desde SQL Server.

    Raises
    ------
    RuntimeError
        Si falla la conexion, el driver o el procedimiento.
    """
    try:
        from sqlalchemy import create_engine, text
        from sqlalchemy.pool import NullPool
    except Exception as exc:
        raise RuntimeError(
            "Faltan dependencias para SQL Server. Instala sqlalchemy y pyodbc. "
            f"Detalle: {exc}"
        ) from exc

    try:
        params = quote_plus(
            f"DRIVER={{{DRIVER}}};"
            f"SERVER={SERVIDOR};"
            f"DATABASE={BASE_DATOS};"
            "Trusted_Connection=yes;"
        )
        url = f"mssql+pyodbc:///?odbc_connect={params}"

        sentencia = text(
            f"EXEC [{BASE_DATOS}].[{ESQUEMA}].[{PROCEDIMIENTO}] "
            f"@ACCION='EDA_BASE', "
            f"@FEC_INI='{FEC_INI}', "
            f"@FEC_FIN='{FEC_FIN}'"
        )

        motor = create_engine(url, echo=False, poolclass=NullPool)

        with motor.connect() as conn:
            resultado = conn.execute(sentencia)
            df = pd.DataFrame.from_records(
                resultado.fetchall(),
                columns=list(resultado.keys()),
            )

    except Exception as exc:
        raise RuntimeError(f"Error al cargar datos desde SQL Server: {exc}") from exc

    if df.empty:
        raise RuntimeError("El procedimiento retorno 0 filas.")

    return df


def cargar_datos() -> pd.DataFrame:
    """
    Carga datos desde SQL Server.

    Parameters
    ----------
    None
        Usa la configuracion global de SQL Server.

    Returns
    -------
    pd.DataFrame
        Dataset original.
    """
    return cargar_datos_sql_server()

## 3. Preparacion del valor unitario

Aqui se calcula el `VALOR_UNITARIO` y se limpian valores imposibles:

- FOB nulo o negativo.
- Peso neto nulo, cero o negativo.
- Valor unitario infinito.
- Subpartida vacia.

In [3]:
# =============================================================================
# 3. PREPARACION DEL DATASET
# =============================================================================

def convertir_numerico_seguro(serie: pd.Series) -> pd.Series:
    """
    Convierte una serie a numerica de forma segura.

    Parameters
    ----------
    serie : pd.Series
        Serie original.

    Returns
    -------
    pd.Series
        Serie convertida a float, con NaN cuando no se puede convertir.
    """
    return pd.to_numeric(serie, errors="coerce")


def preparar_dataset(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Limpia el dataset y calcula el valor unitario.

    Parameters
    ----------
    df_raw : pd.DataFrame
        Dataset original cargado desde CSV o SQL Server.

    Returns
    -------
    pd.DataFrame
        Dataset limpio con columnas canonicas y variables derivadas.

    Raises
    ------
    ValueError
        Si despues de limpiar no quedan registros validos.
    """
    df = estandarizar_columnas(df_raw)

    df[COL_SUBPARTIDA] = df[COL_SUBPARTIDA].astype(str).str.strip()
    df[COL_FOB] = convertir_numerico_seguro(df[COL_FOB])
    df[COL_PESO] = convertir_numerico_seguro(df[COL_PESO])

    if COL_CANTIDAD in df.columns:
        df[COL_CANTIDAD] = convertir_numerico_seguro(df[COL_CANTIDAD])

    if COL_FECHA in df.columns:
        df[COL_FECHA] = pd.to_datetime(df[COL_FECHA], errors="coerce")

    df = df[
        df[COL_SUBPARTIDA].notna()
        & (df[COL_SUBPARTIDA].str.len() > 0)
        & df[COL_FOB].notna()
        & df[COL_PESO].notna()
        & (df[COL_FOB] > 0)
        & (df[COL_PESO] > 0)
    ].copy()

    df[COL_VU] = df[COL_FOB] / df[COL_PESO]
    df = df[np.isfinite(df[COL_VU]) & (df[COL_VU] > 0)].copy()
    df[COL_LOG_VU] = np.log1p(df[COL_VU])

    if df.empty:
        raise ValueError("No quedan registros validos luego de calcular valor unitario.")

    df["ID_REGISTRO_DEMO"] = np.arange(1, len(df) + 1)
    df["ES_SINTETICO"] = 0
    df["TIPO_OUTLIER_SINTETICO"] = "REAL"

    return df.reset_index(drop=True)


def seleccionar_subpartidas_demo(df: pd.DataFrame) -> list[str]:
    """
    Selecciona subpartidas para la demo.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset limpio.

    Returns
    -------
    list[str]
        Lista de subpartidas seleccionadas.

    Notes
    -----
    Si SUBPARTIDAS_DEMO_FORZADAS tiene valores, se usan esas subpartidas.
    Si no, se priorizan capitulos agropecuarios y mayor cantidad de registros.
    """
    if SUBPARTIDAS_DEMO_FORZADAS:
        existentes = set(df[COL_SUBPARTIDA].astype(str))
        seleccionadas = [
            str(spn) for spn in SUBPARTIDAS_DEMO_FORZADAS
            if str(spn) in existentes
        ]
        if seleccionadas:
            return seleccionadas

    df_agro = df[df[COL_SUBPARTIDA].astype(str).str.startswith(CAPITULOS_AGRO_DEMO)].copy()

    if df_agro.empty:
        df_agro = df.copy()

    conteo = (
        df_agro.groupby(COL_SUBPARTIDA)
        .size()
        .reset_index(name="REGISTROS")
        .query("REGISTROS >= @MIN_REGISTROS_SUBPARTIDA")
        .sort_values("REGISTROS", ascending=False)
        .head(N_SUBPARTIDAS_DEMO)
    )

    return conteo[COL_SUBPARTIDA].astype(str).tolist()

## 4. Ejecucion de carga desde SQL Server

Ejecuta esta celda cuando tengas configurada la conexion a SQL Server.  
El notebook llamara al procedimiento configurado y luego seleccionara 3 a 5 subpartidas agropecuarias de alto volumen para la demo.

In [4]:
# =============================================================================
# 4. CARGA Y SELECCION DE SUBPARTIDAS
# =============================================================================

inicio = time.perf_counter()

df_raw = cargar_datos()
df_base = preparar_dataset(df_raw)

subpartidas_demo = seleccionar_subpartidas_demo(df_base)
df_demo = df_base[df_base[COL_SUBPARTIDA].isin(subpartidas_demo)].copy()

resumen_carga = pd.DataFrame({
    "indicador": [
        "registros_base_limpios",
        "subpartidas_base",
        "subpartidas_demo",
        "registros_demo",
        "tiempo_segundos",
    ],
    "valor": [
        f"{len(df_base):,}",
        f"{df_base[COL_SUBPARTIDA].nunique():,}",
        ", ".join(subpartidas_demo),
        f"{len(df_demo):,}",
        f"{time.perf_counter() - inicio:.2f}",
    ],
})

resumen_carga

,indicador,valor
0,registros_base_limpios,"390,426"
1,subpartidas_base,786
2,subpartidas_demo,"1008509000-QUINUA, EXCEPTO PARA LA SIEMBRA, 12..."
3,registros_demo,"9,458"
4,tiempo_segundos,6.53


## 5. Protocolo: folds por subpartida

El split se hace **dentro de cada subpartida**.  
No se mezclan subpartidas porque cada producto tiene su propio universo de precios y unidades.

In [5]:
# =============================================================================
# 5. FOLDS POR SUBPARTIDA
# =============================================================================

def crear_folds_por_subpartida(df: pd.DataFrame) -> list[dict[str, Any]]:
    """
    Crea folds independientes dentro de cada subpartida.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset limpio filtrado para demo.

    Returns
    -------
    list[dict[str, Any]]
        Lista de folds con train, test, fold_id y subpartida.

    Raises
    ------
    ValueError
        Si no se puede crear ningun fold valido.
    """
    folds = []

    for subpartida, df_sub in df.groupby(COL_SUBPARTIDA):
        df_sub = df_sub.copy().reset_index(drop=True)

        if len(df_sub) < MIN_REGISTROS_SUBPARTIDA:
            continue

        n_splits = min(N_FOLDS, len(df_sub))
        if n_splits < 2:
            continue

        kfold = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

        for fold_id, (idx_train, idx_test) in enumerate(kfold.split(df_sub), start=1):
            folds.append({
                "subpartida": str(subpartida),
                "fold_id": int(fold_id),
                "train": df_sub.iloc[idx_train].copy().reset_index(drop=True),
                "test": df_sub.iloc[idx_test].copy().reset_index(drop=True),
            })

    if not folds:
        raise ValueError("No se crearon folds. Revisa MIN_REGISTROS_SUBPARTIDA.")

    return folds


folds = crear_folds_por_subpartida(df_demo)

resumen_folds = pd.DataFrame([
    {
        "subpartida": item["subpartida"],
        "fold": item["fold_id"],
        "n_train": len(item["train"]),
        "n_test": len(item["test"]),
    }
    for item in folds
])

resumen_folds.head(10)

,subpartida,fold,n_train,n_test
0,1005909000-LOS DEMAS MAICES,1,1189,298
1,1005909000-LOS DEMAS MAICES,2,1189,298
2,1005909000-LOS DEMAS MAICES,3,1190,297
3,1005909000-LOS DEMAS MAICES,4,1190,297
4,1005909000-LOS DEMAS MAICES,5,1190,297
5,"1008509000-QUINUA, EXCEPTO PARA LA SIEMBRA",1,3696,924
6,"1008509000-QUINUA, EXCEPTO PARA LA SIEMBRA",2,3696,924
7,"1008509000-QUINUA, EXCEPTO PARA LA SIEMBRA",3,3696,924
8,"1008509000-QUINUA, EXCEPTO PARA LA SIEMBRA",4,3696,924
9,"1008509000-QUINUA, EXCEPTO PARA LA SIEMBRA",5,3696,924


## 6. Data sintetica: outliers conocidos

La data sintetica se agrega solo para validar recuperacion de outliers conocidos.  
No reemplaza a la data real.

Para cada fold:

- `train`: registros reales.
- `test`: registros reales + outliers sinteticos.
- `ES_SINTETICO = 1`: etiqueta controlada solo para evaluacion.

In [6]:
# =============================================================================
# 6. GENERACION DE OUTLIERS SINTETICOS
# =============================================================================

def generar_outliers_sinteticos(
    train_sub: pd.DataFrame,
    test_sub: pd.DataFrame,
    tasa: float = TASA_OUTLIERS_SINTETICOS,
) -> pd.DataFrame:
    """
    Inserta outliers sinteticos en el conjunto de test de una subpartida.

    Parameters
    ----------
    train_sub : pd.DataFrame
        Datos reales de entrenamiento de una subpartida.
    test_sub : pd.DataFrame
        Datos reales de prueba de una subpartida.
    tasa : float, default=TASA_OUTLIERS_SINTETICOS
        Proporcion de outliers sinteticos respecto al tamano del test.

    Returns
    -------
    pd.DataFrame
        Test aumentado con registros sinteticos etiquetados.

    Notes
    -----
    Los outliers se construyen modificando el valor unitario esperado y
    recalculando el FOB para mantener coherencia con el peso neto.
    """
    if test_sub.empty:
        return test_sub.copy()

    n_sinteticos = max(2, int(math.ceil(len(test_sub) * tasa)))
    muestras = test_sub.sample(
        n=min(n_sinteticos, len(test_sub)),
        replace=True,
        random_state=SEED,
    ).copy()

    mediana_vu = float(train_sub[COL_VU].median())
    if not np.isfinite(mediana_vu) or mediana_vu <= 0:
        mediana_vu = float(test_sub[COL_VU].median())

    mitad = len(muestras) // 2
    tipos = np.array(["BAJO"] * mitad + ["ALTO"] * (len(muestras) - mitad))
    RNG.shuffle(tipos)

    factores = np.where(
        tipos == "BAJO",
        FACTOR_OUTLIER_BAJO,
        FACTOR_OUTLIER_ALTO,
    )

    ruido = RNG.uniform(0.85, 1.15, size=len(muestras))
    nuevo_vu = mediana_vu * factores * ruido

    muestras[COL_VU] = nuevo_vu
    muestras[COL_LOG_VU] = np.log1p(muestras[COL_VU])
    muestras[COL_FOB] = muestras[COL_VU] * muestras[COL_PESO]
    muestras["ES_SINTETICO"] = 1
    muestras["TIPO_OUTLIER_SINTETICO"] = tipos
    muestras["ID_REGISTRO_DEMO"] = [
        f"SYN_{i}" for i in range(1, len(muestras) + 1)
    ]

    test_real = test_sub.copy()
    test_real["ES_SINTETICO"] = 0
    test_real["TIPO_OUTLIER_SINTETICO"] = "REAL"

    return pd.concat([test_real, muestras], ignore_index=True)


ejemplo_fold = folds[0]
test_con_sinteticos = generar_outliers_sinteticos(
    ejemplo_fold["train"],
    ejemplo_fold["test"],
)

test_con_sinteticos["ES_SINTETICO"].value_counts().rename("conteo")

ES_SINTETICO
0    298
1     24
Name: conteo, dtype: int64

## 7. Modelos no supervisados

Modelos incluidos para la demo:

1. **IQR por subpartida:** linea base interpretable.
2. **Robust Z por subpartida:** usa mediana y MAD.
3. **LOF novelty:** aprende densidad local con train y predice sobre test.
4. **Isolation Forest:** modelo no supervisado con semilla reproducible.

Todos los parametros se ajustan con `train`; el `test` no se usa para aprender limites.

In [7]:
# =============================================================================
# 7. MODELOS DE DETECCION
# =============================================================================

@dataclass(frozen=True)
class ConfigModelo:
    """
    Configuracion de un detector de outliers.

    Parameters
    ----------
    nombre : str
        Nombre del modelo.
    parametros : dict[str, Any]
        Hiperparametros usados por el modelo.

    Returns
    -------
    ConfigModelo
        Configuracion inmutable para reproducibilidad.
    """
    nombre: str
    parametros: dict[str, Any]


CONFIG_MODELOS = [
    ConfigModelo("IQR_SUBPARTIDA", {"k": 1.5}),
    ConfigModelo("ROBUST_Z_SUBPARTIDA", {"z_umbral": 3.5}),
    ConfigModelo("LOF_NOVELTY", {"n_neighbors": 20, "contamination": 0.08}),
    ConfigModelo("ISOLATION_FOREST", {"contamination": 0.08, "n_estimators": 200}),
]


def predecir_iqr(train: pd.DataFrame, test: pd.DataFrame, k: float = 1.5) -> np.ndarray:
    """
    Detecta outliers usando limites IQR aprendidos en train.

    Parameters
    ----------
    train : pd.DataFrame
        Datos reales de entrenamiento.
    test : pd.DataFrame
        Datos de prueba.
    k : float, default=1.5
        Multiplicador del rango intercuartilico.

    Returns
    -------
    np.ndarray
        Vector binario: 1 si es outlier, 0 si es normal.
    """
    q1 = float(train[COL_VU].quantile(0.25))
    q3 = float(train[COL_VU].quantile(0.75))
    iqr = q3 - q1

    if not np.isfinite(iqr) or iqr <= 0:
        return np.zeros(len(test), dtype=int)

    lim_inf = q1 - k * iqr
    lim_sup = q3 + k * iqr

    return ((test[COL_VU] < lim_inf) | (test[COL_VU] > lim_sup)).astype(int).to_numpy()


def predecir_robust_z(
    train: pd.DataFrame,
    test: pd.DataFrame,
    z_umbral: float = 3.5,
) -> np.ndarray:
    """
    Detecta outliers mediante z robusto aprendido en train.

    Parameters
    ----------
    train : pd.DataFrame
        Datos reales de entrenamiento.
    test : pd.DataFrame
        Datos de prueba.
    z_umbral : float, default=3.5
        Umbral absoluto del z robusto.

    Returns
    -------
    np.ndarray
        Vector binario: 1 si es outlier, 0 si es normal.
    """
    mediana = float(train[COL_VU].median())
    mad = float(np.median(np.abs(train[COL_VU] - mediana)))

    if not np.isfinite(mad) or mad <= 0:
        return np.zeros(len(test), dtype=int)

    z_robusto = 0.6745 * (test[COL_VU] - mediana) / mad
    return (np.abs(z_robusto) > z_umbral).astype(int).to_numpy()


def matriz_features(df: pd.DataFrame) -> np.ndarray:
    """
    Construye matriz de features para modelos multivariados.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset con variables derivadas.

    Returns
    -------
    np.ndarray
        Matriz escalada robustamente por columnas.

    Notes
    -----
    Las variables se mantienen simples para que la demo sea explicable.
    """
    features = df[[COL_LOG_VU, COL_FOB, COL_PESO]].copy()
    features[COL_FOB] = np.log1p(features[COL_FOB])
    features[COL_PESO] = np.log1p(features[COL_PESO])
    return features.to_numpy()


def predecir_lof(train: pd.DataFrame, test: pd.DataFrame, params: dict[str, Any]) -> np.ndarray:
    """
    Detecta outliers con Local Outlier Factor en modo novelty.

    Parameters
    ----------
    train : pd.DataFrame
        Datos reales de entrenamiento.
    test : pd.DataFrame
        Datos de prueba.
    params : dict[str, Any]
        Hiperparametros del modelo.

    Returns
    -------
    np.ndarray
        Vector binario: 1 si es outlier, 0 si es normal.
    """
    n_neighbors = min(int(params["n_neighbors"]), max(2, len(train) - 1))

    scaler = RobustScaler()
    x_train = scaler.fit_transform(matriz_features(train))
    x_test = scaler.transform(matriz_features(test))

    modelo = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination=float(params["contamination"]),
        novelty=True,
    )
    modelo.fit(x_train)
    pred = modelo.predict(x_test)

    return (pred == -1).astype(int)


def predecir_isolation_forest(
    train: pd.DataFrame,
    test: pd.DataFrame,
    params: dict[str, Any],
) -> np.ndarray:
    """
    Detecta outliers con Isolation Forest.

    Parameters
    ----------
    train : pd.DataFrame
        Datos reales de entrenamiento.
    test : pd.DataFrame
        Datos de prueba.
    params : dict[str, Any]
        Hiperparametros del modelo.

    Returns
    -------
    np.ndarray
        Vector binario: 1 si es outlier, 0 si es normal.
    """
    scaler = RobustScaler()
    x_train = scaler.fit_transform(matriz_features(train))
    x_test = scaler.transform(matriz_features(test))

    modelo = IsolationForest(
        n_estimators=int(params["n_estimators"]),
        contamination=float(params["contamination"]),
        random_state=SEED,
    )
    modelo.fit(x_train)
    pred = modelo.predict(x_test)

    return (pred == -1).astype(int)


def predecir_modelo(config: ConfigModelo, train: pd.DataFrame, test: pd.DataFrame) -> np.ndarray:
    """
    Ejecuta un detector segun la configuracion recibida.

    Parameters
    ----------
    config : ConfigModelo
        Configuracion del modelo.
    train : pd.DataFrame
        Datos reales de entrenamiento.
    test : pd.DataFrame
        Datos de prueba.

    Returns
    -------
    np.ndarray
        Vector binario de prediccion de outlier.
    """
    if config.nombre == "IQR_SUBPARTIDA":
        return predecir_iqr(train, test, k=float(config.parametros["k"]))

    if config.nombre == "ROBUST_Z_SUBPARTIDA":
        return predecir_robust_z(
            train,
            test,
            z_umbral=float(config.parametros["z_umbral"]),
        )

    if config.nombre == "LOF_NOVELTY":
        return predecir_lof(train, test, config.parametros)

    if config.nombre == "ISOLATION_FOREST":
        return predecir_isolation_forest(train, test, config.parametros)

    raise ValueError(f"Modelo no reconocido: {config.nombre}")

## 8. Evaluacion con outliers sinteticos

Aqui se mide si el modelo recupera las anomalias insertadas.  
Esta es la parte mas fuerte para justificar metodologicamente el uso de data sintetica.

In [8]:
# =============================================================================
# 8. EVALUACION CON DATA SINTETICA
# =============================================================================

def calcular_metricas_sinteticas(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """
    Calcula metricas contra la etiqueta sintetica conocida.

    Parameters
    ----------
    y_true : np.ndarray
        Etiqueta real controlada: 1 si es sintetico, 0 si es real.
    y_pred : np.ndarray
        Prediccion del modelo: 1 si es outlier, 0 si es normal.

    Returns
    -------
    dict[str, float]
        Precision, recall, f1, tasa de alertas y conteos principales.
    """
    return {
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "tasa_alertas": float(np.mean(y_pred)),
        "n_test_total": int(len(y_true)),
        "n_sinteticos": int(np.sum(y_true)),
        "n_alertas": int(np.sum(y_pred)),
    }


def evaluar_modelos_en_folds(folds: list[dict[str, Any]]) -> pd.DataFrame:
    """
    Evalua todos los modelos en todos los folds por subpartida.

    Parameters
    ----------
    folds : list[dict[str, Any]]
        Folds generados por subpartida.

    Returns
    -------
    pd.DataFrame
        Tabla larga con metricas por modelo, subpartida y fold.
    """
    resultados = []

    for item in folds:
        subpartida = item["subpartida"]
        fold_id = item["fold_id"]
        train = item["train"]
        test = generar_outliers_sinteticos(train, item["test"])

        y_true = test["ES_SINTETICO"].to_numpy(dtype=int)

        for config in CONFIG_MODELOS:
            try:
                y_pred = predecir_modelo(config, train, test)
                metricas = calcular_metricas_sinteticas(y_true, y_pred)
                metricas.update({
                    "subpartida": subpartida,
                    "fold": fold_id,
                    "modelo": config.nombre,
                    "parametros": json.dumps(config.parametros, ensure_ascii=False),
                    "estado": "OK",
                    "error": "",
                })
            except Exception as exc:
                metricas = {
                    "subpartida": subpartida,
                    "fold": fold_id,
                    "modelo": config.nombre,
                    "parametros": json.dumps(config.parametros, ensure_ascii=False),
                    "precision": np.nan,
                    "recall": np.nan,
                    "f1": np.nan,
                    "tasa_alertas": np.nan,
                    "n_test_total": len(test),
                    "n_sinteticos": int(np.sum(y_true)),
                    "n_alertas": np.nan,
                    "estado": "ERROR",
                    "error": str(exc),
                }

            resultados.append(metricas)

    return pd.DataFrame(resultados)


tabla_resultados_folds = evaluar_modelos_en_folds(folds)

tabla_resultados = (
    tabla_resultados_folds
    .groupby(["subpartida", "modelo"], as_index=False)
    .agg(
        precision_prom=("precision", "mean"),
        recall_prom=("recall", "mean"),
        f1_prom=("f1", "mean"),
        f1_std=("f1", "std"),
        tasa_alertas_prom=("tasa_alertas", "mean"),
        n_folds=("fold", "nunique"),
        n_sinteticos_total=("n_sinteticos", "sum"),
    )
    .sort_values(["subpartida", "f1_prom"], ascending=[True, False])
)

tabla_resultados

,subpartida,modelo,precision_prom,recall_prom,f1_prom,f1_std,tasa_alertas_prom,n_folds,n_sinteticos_total
0,1005909000-LOS DEMAS MAICES,IQR_SUBPARTIDA,0.449467,1.000000,0.618049,0.059500,0.168648,5,120
2,1005909000-LOS DEMAS MAICES,LOF_NOVELTY,0.463058,0.866667,0.601706,0.043677,0.141265,5,120
1,1005909000-LOS DEMAS MAICES,ISOLATION_FOREST,0.443568,0.783333,0.562934,0.056017,0.134421,5,120
3,1005909000-LOS DEMAS MAICES,ROBUST_Z_SUBPARTIDA,0.454041,0.533333,0.480546,0.039640,0.092703,5,120
7,"1008509000-QUINUA, EXCEPTO PARA LA SIEMBRA",ROBUST_Z_SUBPARTIDA,0.591060,1.000000,0.742418,0.029564,0.125852,5,370
4,"1008509000-QUINUA, EXCEPTO PARA LA SIEMBRA",IQR_SUBPARTIDA,0.528285,1.000000,0.690973,0.024441,0.140681,5,370
5,"1008509000-QUINUA, EXCEPTO PARA LA SIEMBRA",ISOLATION_FOREST,0.495839,1.000000,0.662422,0.030069,0.150100,5,370
6,"1008509000-QUINUA, EXCEPTO PARA LA SIEMBRA",LOF_NOVELTY,0.461053,0.894595,0.608440,0.035863,0.143888,5,370
10,"1209919000-DEMAS SEMILLAS DE HORTALIZAS, PARA ...",LOF_NOVELTY,0.508004,0.900000,0.636345,0.101714,0.144067,5,50
9,"1209919000-DEMAS SEMILLAS DE HORTALIZAS, PARA ...",ISOLATION_FOREST,0.321097,0.480000,0.378102,0.086843,0.116794,5,50


## 9. Tabla principal para la exposicion

Esta tabla responde a la parte de **resultados** de la demo.  
La lectura debe ser por subpartida: que modelo recupero mejor los outliers sinteticos y con que estabilidad.

In [9]:
# =============================================================================
# 9. TABLA PRINCIPAL
# =============================================================================

def seleccionar_modelo_ganador(tabla: pd.DataFrame) -> pd.DataFrame:
    """
    Selecciona el mejor modelo por subpartida segun F1 promedio.

    Parameters
    ----------
    tabla : pd.DataFrame
        Tabla agregada por subpartida y modelo.

    Returns
    -------
    pd.DataFrame
        Una fila por subpartida con el modelo ganador.
    """
    ordenada = tabla.sort_values(
        ["subpartida", "f1_prom", "recall_prom", "precision_prom"],
        ascending=[True, False, False, False],
    )
    return ordenada.groupby("subpartida", as_index=False).head(1).reset_index(drop=True)


tabla_ganadores = seleccionar_modelo_ganador(tabla_resultados)

tabla_principal = tabla_ganadores.copy()
tabla_principal["comentario_tecnico"] = tabla_principal.apply(
    lambda row: (
        f"En la subpartida {row['subpartida']}, el modelo {row['modelo']} "
        f"recupero outliers sinteticos con recall {row['recall_prom']:.3f} "
        f"y f1 {row['f1_prom']:.3f}. La tasa promedio de alertas fue "
        f"{row['tasa_alertas_prom']:.3f}."
    ),
    axis=1,
)

tabla_principal

,subpartida,modelo,precision_prom,recall_prom,f1_prom,f1_std,tasa_alertas_prom,n_folds,n_sinteticos_total,comentario_tecnico
0,1005909000-LOS DEMAS MAICES,IQR_SUBPARTIDA,0.449467,1.000000,0.618049,0.059500,0.168648,5,120,"En la subpartida 1005909000-LOS DEMAS MAICES, ..."
1,"1008509000-QUINUA, EXCEPTO PARA LA SIEMBRA",ROBUST_Z_SUBPARTIDA,0.591060,1.000000,0.742418,0.029564,0.125852,5,370,"En la subpartida 1008509000-QUINUA, EXCEPTO PA..."
2,"1209919000-DEMAS SEMILLAS DE HORTALIZAS, PARA ...",LOF_NOVELTY,0.508004,0.900000,0.636345,0.101714,0.144067,5,50,En la subpartida 1209919000-DEMAS SEMILLAS DE ...
3,1211903000-OREGANO (ORIGANUM VULGARE),ISOLATION_FOREST,0.486626,0.942857,0.640475,0.098888,0.147312,5,70,En la subpartida 1211903000-OREGANO (ORIGANUM ...
4,1211909099-RAICES DE REGALIZ,ROBUST_Z_SUBPARTIDA,0.547449,0.516129,0.528907,0.040101,0.073149,5,155,"En la subpartida 1211909099-RAICES DE REGALIZ,..."


## 10. Ablaciones: que cambio movio la aguja

Ablacion principal:

- **Modelo global:** calcula limites con todas las subpartidas mezcladas.
- **Modelo por subpartida:** calcula limites dentro de cada `NUM_PARTNANDI`.

Si el modelo por subpartida mejora, el mensaje metodologico es fuerte:

> El cambio que movio la aguja fue respetar que cada subpartida es un universo independiente.

In [10]:
# =============================================================================
# 10. ABLACION GLOBAL VS SUBPARTIDA
# =============================================================================

def evaluar_iqr_global_vs_subpartida(folds: list[dict[str, Any]]) -> pd.DataFrame:
    """
    Compara IQR global contra IQR por subpartida.

    Parameters
    ----------
    folds : list[dict[str, Any]]
        Folds generados por subpartida.

    Returns
    -------
    pd.DataFrame
        Metricas comparativas de la ablacion.
    """
    registros = []

    train_global = pd.concat([item["train"] for item in folds], ignore_index=True)

    for item in folds:
        subpartida = item["subpartida"]
        fold_id = item["fold_id"]
        train_sub = item["train"]
        test = generar_outliers_sinteticos(train_sub, item["test"])
        y_true = test["ES_SINTETICO"].to_numpy(dtype=int)

        pred_global = predecir_iqr(train_global, test, k=1.5)
        met_global = calcular_metricas_sinteticas(y_true, pred_global)
        met_global.update({
            "subpartida": subpartida,
            "fold": fold_id,
            "experimento": "IQR_GLOBAL_MEZCLADO",
        })
        registros.append(met_global)

        pred_sub = predecir_iqr(train_sub, test, k=1.5)
        met_sub = calcular_metricas_sinteticas(y_true, pred_sub)
        met_sub.update({
            "subpartida": subpartida,
            "fold": fold_id,
            "experimento": "IQR_POR_SUBPARTIDA",
        })
        registros.append(met_sub)

    return pd.DataFrame(registros)


tabla_ablaciones_folds = evaluar_iqr_global_vs_subpartida(folds)

tabla_ablaciones = (
    tabla_ablaciones_folds
    .groupby(["experimento"], as_index=False)
    .agg(
        precision_prom=("precision", "mean"),
        recall_prom=("recall", "mean"),
        f1_prom=("f1", "mean"),
        tasa_alertas_prom=("tasa_alertas", "mean"),
    )
    .sort_values("f1_prom", ascending=False)
)

tabla_ablaciones

,experimento,precision_prom,recall_prom,f1_prom,tasa_alertas_prom
1,IQR_POR_SUBPARTIDA,0.467667,0.651226,0.501572,0.108859
0,IQR_GLOBAL_MEZCLADO,0.399075,0.603226,0.376071,0.344597


## 11. Aplicacion sobre datos reales

Luego de validar con sinteticos, aplicamos el modelo ganador por subpartida sobre datos reales.  
El resultado no se interpreta como fraude; se interpreta como **alerta tecnica para revision**.

Nota tecnica: si `tabla_ganadores` no contiene la columna `parametros`, el notebook recupera los parametros desde `CONFIG_MODELOS`.

In [11]:
# =============================================================================
# 11. ALERTAS SOBRE DATOS REALES
# =============================================================================

def obtener_parametros_modelo(nombre_modelo: str) -> dict[str, Any]:
    """
    Obtiene los parametros de un modelo desde CONFIG_MODELOS.

    Parameters
    ----------
    nombre_modelo : str
        Nombre del modelo seleccionado como ganador.

    Returns
    -------
    dict[str, Any]
        Diccionario de parametros del modelo.

    Raises
    ------
    ValueError
        Si el modelo no existe en CONFIG_MODELOS.
    """
    for config in CONFIG_MODELOS:
        if config.nombre == nombre_modelo:
            return dict(config.parametros)

    raise ValueError(f"No se encontraron parametros para el modelo: {nombre_modelo}")


def leer_parametros_fila(fila: pd.Series, nombre_modelo: str) -> dict[str, Any]:
    """
    Lee parametros desde la fila de ganadores o desde CONFIG_MODELOS.

    Parameters
    ----------
    fila : pd.Series
        Fila de la tabla de modelos ganadores.
    nombre_modelo : str
        Nombre del modelo ganador.

    Returns
    -------
    dict[str, Any]
        Parametros del modelo listos para ejecucion.

    Notes
    -----
    Esta funcion corrige el caso donde tabla_ganadores no contiene la columna
    'parametros' porque fue generada desde una tabla agregada por groupby.
    """
    if "parametros" not in fila.index:
        return obtener_parametros_modelo(nombre_modelo)

    valor = fila["parametros"]

    if isinstance(valor, dict):
        return valor

    if pd.isna(valor):
        return obtener_parametros_modelo(nombre_modelo)

    if isinstance(valor, str):
        try:
            return json.loads(valor)
        except json.JSONDecodeError:
            return obtener_parametros_modelo(nombre_modelo)

    return obtener_parametros_modelo(nombre_modelo)


def aplicar_ganadores_a_datos_reales(
    df: pd.DataFrame,
    tabla_ganadores: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplica el modelo ganador por subpartida a datos reales.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset real de demo.
    tabla_ganadores : pd.DataFrame
        Modelo ganador por subpartida.

    Returns
    -------
    pd.DataFrame
        Registros reales marcados como alerta, priorizados por valor unitario.
    """
    alertas = []

    if tabla_ganadores.empty:
        print("tabla_ganadores esta vacia. No se generaron alertas.")
        return pd.DataFrame()

    columnas_obligatorias = {"subpartida", "modelo"}
    faltantes = columnas_obligatorias - set(tabla_ganadores.columns)
    if faltantes:
        raise KeyError(
            f"tabla_ganadores no tiene las columnas obligatorias: {faltantes}. "
            f"Columnas disponibles: {list(tabla_ganadores.columns)}"
        )

    for _, fila in tabla_ganadores.iterrows():
        subpartida = str(fila["subpartida"])
        modelo = str(fila["modelo"])
        params = leer_parametros_fila(fila, modelo)

        df_sub = df[df[COL_SUBPARTIDA].astype(str) == subpartida].copy()
        if len(df_sub) < MIN_REGISTROS_SUBPARTIDA:
            continue

        # En datos reales sin etiquetas, usamos todo el universo de la subpartida
        # como base de aprendizaje descriptiva para generar alertas.
        config = ConfigModelo(modelo, params)
        pred = predecir_modelo(config, df_sub, df_sub)

        df_sub["ES_ALERTA"] = pred
        df_sub["MODELO_GANADOR"] = modelo
        df_sub["RANK_SEVERIDAD_VU"] = df_sub[COL_VU].rank(ascending=False, method="dense")

        alertas.append(df_sub[df_sub["ES_ALERTA"] == 1].copy())

    if not alertas:
        print("No se encontraron alertas con los modelos ganadores.")
        return pd.DataFrame()

    resultado = pd.concat(alertas, ignore_index=True)
    columnas_salida = [
        "ID_REGISTRO_DEMO",
        COL_SUBPARTIDA,
        COL_VU,
        COL_FOB,
        COL_PESO,
        "MODELO_GANADOR",
        "RANK_SEVERIDAD_VU",
    ]

    for col in [COL_FECHA, COL_OPERADOR, COL_ADUANA, COL_CANTIDAD]:
        if col in resultado.columns and col not in columnas_salida:
            columnas_salida.append(col)

    return resultado[columnas_salida].sort_values(
        [COL_SUBPARTIDA, "RANK_SEVERIDAD_VU"],
        ascending=[True, True],
    )


alertas_reales = aplicar_ganadores_a_datos_reales(df_demo, tabla_ganadores)
alertas_reales.head(20)

,ID_REGISTRO_DEMO,NUM_PARTNANDI,VALOR_UNITARIO,FOB_DOLAR,PESO_NETO,MODELO_GANADOR,RANK_SEVERIDAD_VU,ADUANA
99,8368,1005909000-LOS DEMAS MAICES,94.095238,1976.00,21.0,IQR_SUBPARTIDA,1.0,MARITIMA DEL CALLAO
127,39574,1005909000-LOS DEMAS MAICES,94.095238,1976.00,21.0,IQR_SUBPARTIDA,1.0,MARITIMA DEL CALLAO
22,7096,1005909000-LOS DEMAS MAICES,89.142857,1872.00,21.0,IQR_SUBPARTIDA,2.0,MARITIMA DEL CALLAO
24,7117,1005909000-LOS DEMAS MAICES,89.142857,1872.00,21.0,IQR_SUBPARTIDA,2.0,MARITIMA DEL CALLAO
28,7212,1005909000-LOS DEMAS MAICES,88.780488,3640.00,41.0,IQR_SUBPARTIDA,3.0,MARITIMA DEL CALLAO
33,7243,1005909000-LOS DEMAS MAICES,88.780488,3640.00,41.0,IQR_SUBPARTIDA,3.0,MARITIMA DEL CALLAO
6,6348,1005909000-LOS DEMAS MAICES,87.710843,7280.00,83.0,IQR_SUBPARTIDA,4.0,MARITIMA DEL CALLAO
11,6517,1005909000-LOS DEMAS MAICES,86.666667,1820.00,21.0,IQR_SUBPARTIDA,5.0,MARITIMA DEL CALLAO
26,7206,1005909000-LOS DEMAS MAICES,72.833333,437.00,6.0,IQR_SUBPARTIDA,6.0,MARITIMA DEL CALLAO
91,7744,1005909000-LOS DEMAS MAICES,65.714286,460.00,7.0,IQR_SUBPARTIDA,7.0,MARITIMA DEL CALLAO


## 12. Figura clave

Para la exposicion se recomienda mostrar una sola figura fuerte:

**Valor unitario por subpartida, resaltando alertas.**

El mensaje visual debe ser:

> Las subpartidas tienen rangos distintos; por eso el analisis debe hacerse por subpartida.

In [21]:
# =============================================================================
# 12. FIGURA CLAVE SIMPLE
# =============================================================================

import plotly.express as px


def construir_figura_clave(df: pd.DataFrame, alertas: pd.DataFrame, ruta_salida: Path) -> None:
    """
    Construye una figura simple en HTML del valor unitario por subpartida.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset real de demo.
    alertas : pd.DataFrame
        Registros marcados como alerta.
    ruta_salida : Path
        Ruta base donde se guardara el HTML.

    Returns
    -------
    None
        Guarda la figura como archivo HTML.
    """

    base = df.copy()

    base[COL_VU] = pd.to_numeric(base[COL_VU], errors="coerce")

    base = base[
        base[COL_SUBPARTIDA].notna()
        & base[COL_VU].notna()
        & np.isfinite(base[COL_VU])
        & (base[COL_VU] > 0)
    ].copy()

    if base.empty:
        print("No hay datos validos para graficar.")
        return

    base["TIPO_REGISTRO"] = "No alerta"

    if isinstance(alertas, pd.DataFrame) and not alertas.empty:
        if "ID_REGISTRO_DEMO" in alertas.columns and "ID_REGISTRO_DEMO" in base.columns:
            ids_alerta = set(alertas["ID_REGISTRO_DEMO"].astype(str))

            base.loc[
                base["ID_REGISTRO_DEMO"].astype(str).isin(ids_alerta),
                "TIPO_REGISTRO"
            ] = "Alerta"

    fig = px.scatter(
        base,
        x=COL_SUBPARTIDA,
        y=COL_VU,
        color="TIPO_REGISTRO",
        log_y=True,
        title="Valor unitario y alertas por subpartida",
        labels={
            COL_SUBPARTIDA: "Subpartida NUM_PARTNANDI",
            COL_VU: "Valor unitario FOB / Peso neto",
            "TIPO_REGISTRO": "Tipo de registro"
        }
    )

    fig.update_layout(
        xaxis_tickangle=-45,
        height=600,
        width=1100
    )

    ruta_html = ruta_salida.with_suffix(".html")
    ruta_html.parent.mkdir(parents=True, exist_ok=True)

    fig.write_html(str(ruta_html), auto_open=True)

    print(f"Figura guardada en: {ruta_html}")
    print("Se abrio en el navegador. Si no abre, busca el archivo HTML en la carpeta artifacts.")


construir_figura_clave(df_demo, alertas_reales, FIGURA_CLAVE)

Figura guardada en: artifacts_demo_storytelling_sql_server\figura_clave_outliers_por_subpartida.html
Se abrio en el navegador. Si no abre, busca el archivo HTML en la carpeta artifacts.


## 13. Riesgos y plan

Riesgos principales para mencionar en la demo:

| Riesgo | Control |
|---|---|
| Subpartidas con pocos registros | Definir minimo de observaciones |
| Outliers reales que son operaciones validas | Interpretar como alerta, no como fraude |
| Diferencias de unidad comercial | Validar peso, cantidad y unidad |
| Estacionalidad | Incorporar mes en iteraciones posteriores |
| Falta de etiquetas reales | Usar validacion sintetica + revision experta |
| Sensibilidad de modelos | Comparar varios metodos y folds |

## 14. Reproducibilidad

La demo debe dejar artefactos:

- `resultados_modelos_por_subpartida.csv`
- `ablaciones_global_vs_subpartida.csv`
- `alertas_reales_priorizadas.csv`
- `README_REPRODUCIBILIDAD.txt`
- `figura_clave_outliers_por_subpartida.png`

Todos los CSV se exportan con delimitador `|`.

In [22]:
# =============================================================================
# 14. EXPORTACION DE ARTEFACTOS Y README
# =============================================================================

def exportar_csv_pipe(df: pd.DataFrame, ruta: Path) -> None:
    """
    Exporta un DataFrame con delimitador pipe.

    Parameters
    ----------
    df : pd.DataFrame
        Tabla a exportar.
    ruta : Path
        Ruta de salida.

    Returns
    -------
    None
        Guarda el archivo en disco.
    """
    df.to_csv(ruta, sep="|", index=False, encoding="utf-8")


exportar_csv_pipe(tabla_resultados, RESULTADOS_CSV)
exportar_csv_pipe(tabla_ablaciones, ABLACIONES_CSV)
exportar_csv_pipe(alertas_reales, ALERTAS_CSV)

readme = f'''
DEMO 10-12 MIN - STORYTELLING TECNICO

Fecha de ejecucion: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
Seed: {SEED}
Fuente de datos SQL Server: {FUENTE_DATOS}
Periodo configurado: {FEC_INI} a {FEC_FIN}

Regla metodologica:
1. La base se divide primero por NUM_PARTNANDI.
2. Los folds se crean dentro de cada subpartida.
3. Los parametros se ajustan con train.
4. El test se usa para evaluar.
5. Los outliers sinteticos se insertan solo para validar recuperacion conocida.
6. Las alertas reales no se interpretan como fraude, sino como priorizacion para revision.

Subpartidas usadas en demo:
{json.dumps(subpartidas_demo, indent=2, ensure_ascii=False)}

Modelos:
{json.dumps([asdict(cfg) for cfg in CONFIG_MODELOS], indent=2, ensure_ascii=False)}

Artefactos:
- {RESULTADOS_CSV}
- {ABLACIONES_CSV}
- {ALERTAS_CSV}
- {FIGURA_CLAVE}

Comando sugerido:
jupyter notebook demo_storytelling_subpartida_outliers.ipynb

MLflow:
Opcional. Si se usa, registrar parametros, metricas promedio por modelo y artefactos generados.
'''

README_TXT.write_text(readme.strip(), encoding="utf-8")

print("Artefactos exportados:")
print(RESULTADOS_CSV)
print(ABLACIONES_CSV)
print(ALERTAS_CSV)
print(README_TXT)
print(FIGURA_CLAVE)

Artefactos exportados:
artifacts_demo_storytelling_sql_server\resultados_modelos_por_subpartida.csv
artifacts_demo_storytelling_sql_server\ablaciones_global_vs_subpartida.csv
artifacts_demo_storytelling_sql_server\alertas_reales_priorizadas.csv
artifacts_demo_storytelling_sql_server\README_REPRODUCIBILIDAD.txt
artifacts_demo_storytelling_sql_server\figura_clave_outliers_por_subpartida.png


## 15. MLflow opcional

Esta celda es opcional. Si tienes MLflow instalado, registra el experimento.  
Si no lo tienes instalado, el notebook continua sin romperse.

In [23]:
# =============================================================================
# 15. REGISTRO OPCIONAL CON MLFLOW
# =============================================================================

try:
    import mlflow

    mlflow.set_experiment("tesis_outliers_valor_unitario_demo")

    with mlflow.start_run(run_name="demo_storytelling_subpartida"):
        mlflow.log_param("seed", SEED)
        mlflow.log_param("n_folds", N_FOLDS)
        mlflow.log_param("min_registros_subpartida", MIN_REGISTROS_SUBPARTIDA)
        mlflow.log_param("tasa_outliers_sinteticos", TASA_OUTLIERS_SINTETICOS)
        mlflow.log_param("subpartidas_demo", ",".join(subpartidas_demo))

        for _, row in tabla_ablaciones.iterrows():
            prefijo = row["experimento"].lower()
            mlflow.log_metric(f"{prefijo}_f1", float(row["f1_prom"]))
            mlflow.log_metric(f"{prefijo}_recall", float(row["recall_prom"]))

        mlflow.log_artifact(str(RESULTADOS_CSV))
        mlflow.log_artifact(str(ABLACIONES_CSV))
        mlflow.log_artifact(str(ALERTAS_CSV))
        mlflow.log_artifact(str(README_TXT))
        mlflow.log_artifact(str(FIGURA_CLAVE))

    print("Registro MLflow completado.")
except Exception as exc:
    print("MLflow no se ejecuto. Motivo:")
    print(exc)

MLflow no se ejecuto. Motivo:
No module named 'mlflow'
